In [2]:
import sys
# print(pd.__version__)
print(sys.executable)

c:\Users\brije\OneDrive\Documents\Learning\insurance-agency-ai-analyst\.venv\Scripts\python.exe


Step 1: set up paths and read only the header

In [3]:
import csv
import sqlite3
from pathlib import Path

CSV_PATH = Path("../data/raw/finalapi.csv")
DB_PATH = Path("../database/insurance.db")

In [4]:
with CSV_PATH.open(newline="", encoding="utf-8") as f:
    header = next(csv.reader(f))

In [5]:
print(len(header))
print(header[:5])

49
['AGENCY_ID', 'PRIMARY_AGENCY_ID', 'PROD_ABBR', 'PROD_LINE', 'STATE_ABBR']


Step 2: create the table and stream the rows in

In [6]:
DB_PATH.parent.mkdir(exist_ok=True)
conn = sqlite3.connect(DB_PATH)

In [7]:
columns_sql = ", ".join(f'"{col}" TEXT' for col in header)
conn.execute("DROP TABLE IF EXISTS stg_agency_performance")
conn.execute(f"CREATE TABLE stg_agency_performance ({columns_sql})")

In [8]:
placeholders = ", ".join("?" for _ in header)
with CSV_PATH.open(newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)  # skip header row
    conn.executemany(
        f"INSERT INTO stg_agency_performance VALUES ({placeholders})", reader
    )
conn.commit()

Step 3: check the load

In [9]:
print(conn.execute("SELECT COUNT(*) FROM stg_agency_performance").fetchone())
print(conn.execute("SELECT typeof(WRTN_PREM_AMT), WRTN_PREM_AMT FROM stg_agency_performance LIMIT 1").fetchone())

(213328,)
('text', '3074.63')


In [10]:
print(conn.execute("SELECT * FROM stg_agency_performance limit 10").fetchone())

('3', '3', 'BOILERMACH', 'CL', 'IN', '2005', '0', '0', '0', '40', '3074.63', '149', '3005.77', '0', '8', '99999', '0', '99999', '99999', '1957', '14', '85', '48', 'N', 'Unknown', '99999', '99999', '99999', '99999', '99999', '99999', '99999', '99999', '0', '1', '0', '0', '0', '0', '0', '0', '0', '103', '50', '288', '0', '0', '0', '0')
